# Connecting Two Ways, psycopg2 and SQLAlchemy to pandas

**Module 4: Python Data Analysis**

**Learning Objectives:**
- Open a real PostgreSQL connection with `psycopg2` and load the rows into a pandas DataFrame
- Run the same query through SQLAlchemy and confirm both paths return the same DataFrame
- Write connection code that is safe to commit, with no typed-in password and no connection left open

**Skills:** connection, cursor, driver, engine, environment variable


> **You are not presenting this notebook to students.** Use it as your own answer key while you fill in a student notebook live, in front of the class.


## Scenario

You are back on the city agency's data team, and the CSV export you worked from last class is already stale. The service requests you need live in a PostgreSQL table called `service_requests`, updated all day long. Today you stop exporting and start connecting: you open a real connection from Python, pull the same NYC 311 rows straight out of the live table into a pandas DataFrame, do it a second time through a different library, and prove the two agree.


## Before You Start

Last class a date came out of a CSV as plain text. If those same rows arrive over a live database connection instead, does the type still get lost?

*Hint: A CSV file stores only characters. Ask yourself what else a database knows about a column that a file never had a place to write down.*


## Documentation

- A Cursor Is What You **Send SQL Through**: https://www.psycopg.org/docs/usage.html

- Tuples In, a **Real DataFrame** Out: https://www.psycopg.org/docs/cursor.html#cursor.description

- The Same Question, in **One Line of pandas**: https://pandas.pydata.org/docs/reference/api/pandas.read_sql.html

- Open It in a `try`, and **Always** Close It: https://www.psycopg.org/docs/connection.html#connection.close


## A Cursor Is What You **Send SQL Through**

*Context: `service_requests` is the same NYC 311 data as last class, this time living in a real PostgreSQL table instead of a file.*

**Instructor notes:**

> [ACTIVITY] Everyone runs this cell and confirms they see one tuple printed before anyone moves on. Online: thumbs-up reaction rather than reading names out.
> 
> [SCRIPT] Go line by line, because every one of these four lines is a concept, not syntax. `os.environ["DATABASE_URL"]` reads a value your computer is holding outside this file, which is objective three arriving early on purpose. `psycopg2.connect()` opens the real link. `conn.cursor()` gives you the thing you actually send SQL through. `cur.execute()` sends it, and `cur.fetchall()` brings every row back.
> 
> [SCRIPT] Point at the printed output before saying anything else about it. It is a plain Python tuple. No column names anywhere. Ask the room what is missing compared to a DataFrame, and let them notice it themselves rather than telling them.
> 
> [SCRIPT] Say the honest thing about the URL: today everyone points at one shared class database so the room stays in sync. On the project you will point the exact same variable at your own database, and not one line of this code changes.
> 
> [MISCONCEPTIONS] People merge `connect` and `cursor` into one idea. Separate them out loud: the connection is the open line to the server, the cursor is one conversation happening on that line. You can run several cursors on one connection, which is exactly why they are two objects and not one.
> 
> [MISCONCEPTIONS] Somebody will ask why the query is a plain string rather than something more Python-shaped. Answer it directly and briefly: `psycopg2` is a driver, so it hands your SQL to PostgreSQL essentially untouched. That is the whole point of a driver, and it is also the first real difference from the library coming up next.

*Input:*


In [ ]:
import os
import psycopg2

conn = psycopg2.connect(os.environ["DATABASE_URL"])
cur = conn.cursor()

sql = "SELECT * FROM service_requests LIMIT 5"
cur.execute(sql)
rows = cur.fetchall()
print(rows[0])


## Tuples In, a **Real DataFrame** Out

**Instructor notes:**

> [ACTIVITY] Run it, then have the room read the second printed line out loud together. That one line is the payoff for the whole first half of the lesson.
> 
> [SCRIPT] Do the DataFrame construction first. `cur.description` is the cursor's own record of the query it just ran, one entry per column, and the first item in each entry is that column's name. `d[0] for d in cur.description` is just "give me each column's name." Then `pd.DataFrame(rows, columns=cols)` puts values and names back together.
> 
> [SCRIPT] Now the payoff, and slow all the way down here. Last class, `created_date` came out of a CSV as text and needed `pd.to_datetime()` to become a real date. Read the printed dtype out loud. It is a real datetime, and nobody converted anything. The type survived because a connection carries the schema and a file cannot. That is the answer to today's warm-up question, arriving as running code rather than as a claim.
> 
> [SCRIPT] Then the two closing lines. Cursor first, then connection. Say plainly that this is the step the case study was about, and that the next section makes it bulletproof rather than merely remembered.
> 
> [MISCONCEPTIONS] `cur.description` reads like documentation, so people assume it is a docstring or a help message. It is not. It is structured metadata about the last query the cursor ran, and it is the standard way every Python database library exposes column names. Two precisions if anyone digs: it always describes the most recent query, not all of them, and the second item in each entry is a database-specific type code, not a Python type, so do not let anyone read it as a `dtype`.
> 
> [MISCONCEPTIONS] A few people will see text columns print as `object` rather than `str` and think something broke. Nothing broke. Older pandas labels a text column `object`; newer pandas labels it `str`. Both mean text, and the column everyone should actually be looking at today is `created_date`.
> 
> [SOURCES] Verified by running this exact code against a real PostgreSQL 16 table holding this module's own NYC 311 sample: `created_date` comes back as `datetime64[us]`, `unique_key` as `int64`. Column names via `cursor.description` are standard Python DBAPI behavior: PEP 249 defines `.description` as a sequence of seven-item sequences, one per result column, of which `name` and `type_code` are the two mandatory items (peps.python.org/pep-0249/).

*Input:*


In [ ]:
import pandas as pd

cols = [d[0] for d in cur.description]
df = pd.DataFrame(rows, columns=cols)

print(df.head(3))
print(df["created_date"].dtype)

cur.close()
conn.close()


### 🤔 Check for Understanding: Where Column Names Live

*Mode: Poll*

`cur.fetchall()` handed back plain tuples with no column names attached. So where did the DataFrame's column names come from?

- `cur.description`, the cursor's own record of the last query
- pandas guessed them by looking at the values
- They were still attached to the tuples, just hidden

**Answer:** `cur.description`, the cursor's own record of the last query — Every Python database cursor keeps a `description` of the query it just ran, one entry per column, and the first item in each entry is that column's name. `fetchall()` returns only values, so putting the names back on is a separate step you do yourself. That extra step is the price of using a raw driver, and it is exactly what the next section removes.

**Instructor notes:**

> [ACTIVITY] Poll: read all three options, count hands on each, then reveal. Online: have everyone type A, B, or C in chat before anyone explains. Say the letters, not numbers, since the slide itself labels the options A, B, and C. 45 seconds.
> 
> [SCRIPT] Option two is the interesting wrong answer, not option three. pandas genuinely does guess plenty of things from values, including every column's type. It just cannot guess a name, because a name is not in the data. Say that out loud, because it is the same idea as last class's file-versus-schema point wearing different clothes.


## The Same Question, in **One Line of pandas**

**Instructor notes:**

> [ACTIVITY] Everyone runs it and calls out what the last line printed. You want to hear `True` from the whole room. If anyone gets `False`, they almost certainly ran a different query than the `sql` variable from earlier, which is worth surfacing rather than skipping.
> 
> [SCRIPT] Count the lines out loud against the previous two slides. Four steps became one call, and `pd.read_sql` did the fetching and the column names together. Then name what the `with` block bought: the connection is handed back automatically at the end of the block, so there is no close to forget here.
> 
> [SCRIPT] The last printed line is the whole point of the lesson title. `df.equals(df2)` is not comparing values loosely. It checks shape, column names, and every column's type. Getting `True` means the two libraries produced genuinely identical DataFrames, including the real datetime column, not just similar-looking output.
> 
> [SCRIPT] Say the URL detail once, because it will look like magic otherwise: the exact same `DATABASE_URL` string worked for both libraries. A URL starting `postgresql://` with no driver named defaults to `psycopg2`, so this line is quietly using the library from the first half of the lesson. Worth one extra clause: that default only works because `psycopg2` is installed, which is why plenty of teams write the driver into the URL explicitly.
> 
> [MISCONCEPTIONS] People will want to hand `pd.read_sql` the bare `psycopg2` connection from earlier instead of a SQLAlchemy connection. It appears to work, and then prints a warning saying pandas only supports a SQLAlchemy connectable, a database string, or a sqlite3 connection, and that other objects are not tested. Say what pandas actually does next, because it is the concrete reason to care: it falls back to its SQLite code path and runs your Postgres connection through that. Demonstrate it if there is time, because "it ran and only warned" is exactly the kind of thing people learn to scroll past.
> 
> [SOURCES] Verified by running this exact code against a real PostgreSQL 16 table holding this module's own NYC 311 sample, with SQLAlchemy 2.0 and pandas: `df.equals(df2)` printed `True`, and passing a raw psycopg2 connection to `pd.read_sql` emitted pandas' documented UserWarning about untested DBAPI2 objects. pandas' own `read_sql` reference states that if a DBAPI2 object is passed, "only sqlite3 is supported" (pandas.pydata.org/docs/reference/api/pandas.read_sql.html), and pandas' source falls back to its SQLite handler after warning. SQLAlchemy's own docs confirm the driverless `postgresql://` form defaults to psycopg2 (docs.sqlalchemy.org/en/20/core/engines.html).

*Input:*


In [ ]:
from sqlalchemy import create_engine

engine = create_engine(os.environ["DATABASE_URL"])

with engine.connect() as connection:
    df2 = pd.read_sql(sql, connection)

print(df2.head(3))
print("Same as psycopg2?", df.equals(df2))


### 🤔 Check for Understanding: Two Paths, One DataFrame

*Mode: Whiteboard / Collaborative Board*

On your board, draw both paths from the same SQL string to the same DataFrame. Mark every place the two genuinely differ.

**Instructor notes:**

> [ACTIVITY] Whiteboard, in pairs, five minutes. In person: one shared board or a sheet of paper per pair, then two pairs hold theirs up and walk it. Online: a shared Padlet, Jamboard, or FigJam board, one frame per pair, then screen-share two of them. Ask for drawings rather than sentences, but say out loud that a written or spoken step-by-step list of the two paths counts exactly the same as a drawing, so nobody is shut out of an analyze-level beat by the format. When a pair presents, have them narrate what they drew rather than only pointing at it.
> 
> [SCRIPT] Steer toward the differences that are actually in the code they just typed, not toward opinions. Real differences to fish for: `psycopg2` needs a cursor and SQLAlchemy does not expose one here; the column names have to be recovered by hand on one side and arrive attached on the other; closing is manual on one side and automatic inside the `with` block on the other. Real sameness to name: the SQL string is identical, the connection URL is identical, the resulting DataFrame is identical, and SQLAlchemy is using `psycopg2` under there the whole time.
> 
> [SCOPE] Do not let this turn into "so which one should we use." That is genuinely next class's entire lesson, with real scenarios to judge against. Say that out loud and park the question rather than answering it thinly today.
> 
> [MISCONCEPTIONS] Several boards will show SQLAlchemy as a completely separate road to the database. Correct it while walking the room, and then draw the authoritative version on the shared board during the debrief rather than only correcting it pair by pair: your code, then SQLAlchemy, then `psycopg2`, then PostgreSQL, stacked, not side by side. That is the same picture the driver slide named earlier, and seeing it drawn once correctly after trying to draw it themselves is what makes it stick.


## Open It in a `try`, and **Always** Close It

**Instructor notes:**

> [ACTIVITY] Fill in the blank, 30 seconds, before advancing to the answer. In person: someone says both missing pieces out loud. Online: everyone types them in chat at the same time on a count of three. This slide is deliberately the one cell with no `starter_code`, so the student notebook shows the exact same two blanks the slide does rather than a different exercise.
> 
> [SCRIPT] Heads up on the error anyone gets if they run the cell before filling it in: a `try` with nothing after it is a `SyntaxError`, not a runtime error, so Python refuses to run the cell at all and points at the wrong line. That is not a broken notebook. Say it before anyone panics, and note that the error message names exactly what is missing.
> 
> [SCRIPT] After the reveal, explain why `finally` and not just a close at the end. A query can raise. A network can drop. If the close sits on the last line and the line above it throws, the close never runs and the connection is stranded. `finally` runs either way, which is exactly what "always" means here.
> 
> [SCRIPT] Point out that the cursor gets a `with` block and the connection does not. That asymmetry is deliberate and it is the single most useful thing on this slide, so do not rush it.
> 
> [MISCONCEPTIONS] This is the big one, and it catches experienced people too. Writing `with conn:` around the work does not close the connection. In `psycopg2`, a connection used as a context manager commits or rolls back the transaction when the block ends, and then leaves the connection open on purpose so you can reuse it. You still have to call `conn.close()` yourself. If there is time, prove it live: run a `with conn:` block, then print `conn.closed` and watch it print zero. `psycopg2`'s own documentation calls this out under a warning heading, and its successor library's migration guide describes the old behavior as "surprising," so nobody should feel slow for missing it.
> 
> [MISCONCEPTIONS] Someone will argue the connection closes when the script ends anyway. For a small script that finishes, that is broadly true. It stops being true the moment the code lives in a notebook that stays open for hours, a scheduled job that loops, or anything running as a service, which is most real analyst code. Do not teach a habit that only works in the easiest case.
> 
> [SOURCES] Verified by running it: after a `with conn:` block completed normally, `conn.closed` was still `0`, and only an explicit `conn.close()` changed it to `1`. psycopg2's own docs state it under a Warning heading: "exiting the connection's `with` block doesn't close the connection, but only the transaction associated to it. If you want to make sure the connection is closed after a certain point, you should still use a try-catch block" (psycopg.org/docs/usage.html). Scope note for anyone who reads ahead: the successor library, psycopg version 3, deliberately changed this, and `with connection` there does close the connection (psycopg.org/psycopg3/docs/basic/from_pg2.html). This lesson uses psycopg2, which is what the project's own environment installs.

*Input:*


In [ ]:
conn = psycopg2.connect(os.environ["DATABASE_URL"])

try:
    with conn.cursor() as cur:
        cur.execute(sql)
        rows = cur.fetchall()
finally:
    conn.close()

print(len(rows), "rows, connection closed")
